# Project evaluation: Human-Centered Evaluation of Your Collaborative Agent

## **Important! Make a copy of this file...**
- [ ] This is a Colab template that you should not modify.
- [ ] Please make your own copy, by selecting e.g. `File > Save a copy in Drive`.
- [ ] Afterwards, make your file identifiable by renaming it to be `AIAgents-S26-youragenthere.ipynb`.


## Overview

In this assignment, you will evaluate your team project agent using two complementary methods:

- Intrinsic Evaluation — an automated evaluation (e.g., LLM-as-a-judge or code-based metric).
- Extrinsic Evaluation — a small-scale user study with real participants outside the class.

You will design, implement, and analyze these evaluations, then summarize your work in a structured write-up that includes both your results and reflections.

You will also propose one additional evaluation type you could run in the future (but won’t conduct for this assignment).

## Learning Objectives:
By the end of this assignment, you should be able to:

- Apply intrinsic metrics for automated performance measurement of conversational agents.
- Design and conduct a small-scale user study to capture user experience and task outcomes.
- Use the SPHERE Evaluation Card to communicate your evaluation design clearly.
- Draw evidence-based conclusions about an agent’s performance.

## Grading Rubric (Total: 100 points)

| Category | Points | Description |
|----------|--------|-------------|
| **Intrinsic Evaluation Design & Implementation** | 30 | 1–3 well-defined metrics tied to agent goals; correct LLM/code implementation; tested on ≥3 diverse examples with meaningful results. |
| **Extrinsic Evaluation Design & Execution** | 30 | Clear, realistic protocol; relevant participants/tasks; both quantitative and qualitative data collected and documented. |
| **SPHERE Evaluation Cards** | 5 | Two completed JSON cards (intrinsic & extrinsic); accurate, detailed, and aligned with evaluation methods. |
| **Results & Conclusions** | 20 | Clear presentation of results; at least one supported conclusion per evaluation with convincing evidence. |
| **Proposed Additional Evaluation** | 10 | Thoughtful and relevant; clear design outline and strong justification for future use. |
| **Clarity & Effort** | 5 | Well-organized, clearly written submission demonstrating strong effort and attention to detail. |




## Step 1: Setup

(You'll need some kind of setup code in your repo, here is an example)

Here, we will do a similar round of setup as in Assignment 1, installing necessary packages for modeling calling. It also copies over the interactive interface I hacked in A1.
**If there are code you would like to reuse, please feel free to copy over here.**

In [ ]:
# same setup as in assignment 1
# this installs a specific earlier version of the library to get the hack
# for the GradioUI chat interface
!pip install 'smolagents[litellm] @ git+https://github.com/huggingface/smolagents.git@df2757b'
!pip install 'smolagents[gradio] @ git+https://github.com/huggingface/smolagents.git@df2757b'

!pip install 'boto3'
!pip install 'markdownify'

  Cloning https://github.com/huggingface/smolagents.git (to revision df2757b) to /tmp/pip-install-5rpalh93/smolagents_d65dc64b28d74b0986ed42307bb356d3
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/smolagents.git /tmp/pip-install-5rpalh93/smolagents_d65dc64b28d74b0986ed42307bb356d3
  Running command git checkout -q df2757b
  Resolved https://github.com/huggingface/smolagents.git to commit df2757b
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 76.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 14.1 MB/s eta 0:00:00
  Created wheel for smolagents: filename=smolagents-1.22.0.dev0-py3-none-any.whl size=141734 sha256=43ac2a7a392dbe8f9d071e6d423e0ac6d6296e8ecd959591954766fe0428942c
  Stored in directory: /tmp/pip-ephem-wheel-cache-5nbbsf1h/wheels/03/e4/7b/da95c516518b730dbbaa66f5e5

In [ ]:
# same interface as A1

from smolagents.gradio_ui import GradioUI, pull_messages_from_step

import os
import re
import shutil
from pathlib import Path
from typing import Generator

from smolagents.agent_types import AgentAudio, AgentImage, AgentText
from smolagents.agents import MultiStepAgent, PlanningStep
from smolagents.memory import ActionStep, FinalAnswerStep, TaskStep
from smolagents.models import ChatMessageStreamDelta, MessageRole, agglomerate_stream_deltas
from smolagents.utils import _is_package_available


class GradioUIChat(GradioUI):
  def create_app(self):
    import gradio as gr
    with gr.Blocks(theme="ocean", fill_height=True) as demo:
      # Add session state to store session-specific data
      session_state = gr.State({})
      stored_messages = gr.State([])
      file_uploads_log = gr.State([])


      with gr.Sidebar():
        gr.Markdown(
          f"# {self.name.replace('_', ' ').capitalize()}"
          "\n> This web ui allows you to interact with a `smolagents` agent that can use tools and execute steps to complete tasks."
          + (f"\n\n**Agent description:**\n{self.description}" if self.description else "")
        )
        # If an upload folder is provided, enable the upload feature
        if self.file_upload_folder is not None:
          upload_file = gr.File(label="Upload a file")
          upload_status = gr.Textbox(label="Upload Status", interactive=False, visible=False)
          upload_file.change(
            self.upload_file,
            [upload_file, file_uploads_log],
            [upload_status, file_uploads_log],
          )

          gr.HTML(
            "<br><br><h4><center>Powered by <a target='_blank' href='https://github.com/huggingface/smolagents'><b>smolagents</b></a></center></h4>"
          )


      # Main chat interface
      chatbot = gr.Chatbot(
        label="Agent",
        type="messages",
        avatar_images=(
          None,
          "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/smolagents/mascot_smol.png",
        ),
        resizeable=True,
        scale=1,
        latex_delimiters=[
          {"left": r"$$", "right": r"$$", "display": True},
          {"left": r"$", "right": r"$", "display": False},
          {"left": r"\[", "right": r"\]", "display": True},
          {"left": r"\(", "right": r"\)", "display": False},
        ],
      )
      with gr.Group():
        text_input = gr.Textbox(
            lines=2,
            label="Chat Message",
            container=False,
            placeholder="Enter your prompt here and press Shift+Enter or press the button",
        )
        submit_btn = gr.Button("Submit")

      text_input.submit(
        self.log_user_message,
        [text_input, file_uploads_log],
        [stored_messages, text_input, submit_btn],
      ).then(
          self.interact_with_agent, [stored_messages, chatbot, session_state], [chatbot]
      ).then(
        lambda: (
          gr.Textbox(interactive=True, placeholder="Enter your prompt here and press Shift+Enter or the button"),
          gr.Button(interactive=True),
        ),
        None,
        [text_input, submit_btn],
      )

      submit_btn.click(
        self.log_user_message,
        [text_input, file_uploads_log],
        [stored_messages, text_input, submit_btn],
      ).then(
          self.interact_with_agent, [stored_messages, chatbot, session_state], [chatbot]
      ).then(
        lambda: (
          gr.Textbox(interactive=True, placeholder="Enter your prompt here and press Shift+Enter or the button"),
          gr.Button(interactive=True),
        ),
        None,
        [text_input, submit_btn],
      )
      chatbot.clear(self.agent.memory.reset)
      return demo

  def stream_to_gradio(
      agent,
      task: str,
      task_images: list | None = None,
      reset_agent_memory: bool = False,
      additional_args: dict | None = None,
  ) -> Generator:
    """Runs an agent with the given task and streams the messages from the agent as gradio ChatMessages."""
    if not _is_package_available("gradio"):
      raise ModuleNotFoundError(
          "Please install 'gradio' extra to use the GradioUI: `pip install 'smolagents[gradio]'`"
          )
    accumulated_events: list[ChatMessageStreamDelta] = []
    if reset_agent_memory:
      agent.memory.reset()
      agent.monitor.reset()
    final_answer = None
    def stream_event(event):
      if isinstance(event, ActionStep | PlanningStep | FinalAnswerStep):
        for message in pull_messages_from_step(
            event,
            # If we're streaming model outputs, no need to display them twice
            skip_model_outputs=getattr(agent, "stream_outputs", False),
        ):
          yield message
        accumulated_events = []
      elif isinstance(event, ChatMessageStreamDelta):
        accumulated_events.append(event)
        text = agglomerate_stream_deltas(accumulated_events).render_as_markdown()
        yield text

      # run the agent only until there's a user input
      step_number = 1
      agent.memory.steps.append(TaskStep(task=task, task_images=[]))
      stream_event(agent.memory.steps[-1])
      while final_answer is None and step_number <= agent.max_steps:
        step = ActionStep(step_number=step_number)
        # Run one step
        final_answer = agent.step(step)
        agent.memory.steps.append(step)
        step_number += 1
        stream_event(agent.memory.steps[-1])
        print(str([tc.dict() for tc in step.tool_calls]))
        if isinstance(step, ActionStep) and \
          step.tool_calls and \
          ("user_input_tool" in str([tc.dict() for tc in step.tool_calls]) or "user_input_tool" in step.code_action):
          break



## Step 2: Load the agent to test

(This is just an example, you will probably want to write this in your project repo as its own script, just document what scripts to run and commit outputs from them and link the files here/in your project Readme)

Here, you will set up your AWS and Huggingface Access Keys to reload the agent you created in Assignment 1. This will be the agent you test.

In [ ]:
import os
# set the keys to be in the env

from google.colab import userdata
os.environ['AWS_ACCESS_KEY_ID'] = userdata.get('AWS_ACCESS_KEY_ID')
os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('AWS_SECRET_ACCESS_KEY')
os.environ["AWS_REGION_NAME"] = "us-east-1"

In [ ]:
import os
from huggingface_hub import login
from smolagents import CodeAgent

# If you forgot how to get the tokens, check back at A1
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
login(token=os.getenv("HF_TOKEN"))

# Use the agent you created and uploaded in A1.
AGENT_PATH ="fill this in here i.e. username/travel_agent"
collab_agent = CodeAgent.from_hub(AGENT_PATH, trust_remote_code=True)
# make sure you succeeded
collab_agent.run("Hello there!")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

╭────────────────────────────────────────── New run - travel_assistant ───────────────────────────────────────────╮
│                                                                                                                 │
│ Hello there!                                                                                                    │
│                                                                                                                 │
╰─ LiteLLMModel - bedrock/us.deepseek.r1-v1:0 ────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
                                                                                                                   
                                                                                                                   
Thought: The user greeted me, so I should respond politely and ask how I can assist them. Since this is the first  
message, there's no conversation history to analyze yet. I'll use the `user_input_tool` to send a friendly response
while maintaining an open question format.                                                                         
                                                                                                                   
<code>                                                                                                             
response = user_input_tool(                                                                                        
    context="Welcome message to start conversation",                                                               
    question="Hello! How can I assist you today?"                                                                  
)                                                                                                                  
final_answer(response)                                                                                             
</code>                                                                                                            

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  response = user_input_tool(                                                                                      
      context="Welcome message to start conversation",                                                             
      question="Hello! How can I assist you today?"                                                                
  )                                                                                                                
  final_answer(response)                                                                                           
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Welcome message to start conversation 
 Hello! How can I assist you today?

[Step 1: Duration 2.24 seconds| Input tokens: 2,452 | Output tokens: 305]

'Welcome message to start conversation \n Hello! How can I assist you today?'

In [ ]:
# You can also try the interface interaction

collab_agent.memory.reset()
collab_agent.monitor.reset()
GradioUIChat(collab_agent).launch(False)

## Step 3: Intrinsic evaluation

- Define **1–3 metrics** that capture an important aspect of your agent’s performance (e.g., factual accuracy, task completeness, response brevity, trade-off clarity).
- Implement these metrics using either:
  - **LLM-as-a-judge** (provide prompt & scoring rubric to an LLM, return numerical score)
  - **Code-based scoring** (string matching, word count, time to completion, etc.)
- Test the metrics on **at least 3 example interactions** with your agent
  - This one will require you to interact with your agent for at least three times, ideally with you simulating diverse personas to create diverse use cases -- Think of how they would represent different use scenarios or edge cases!
  - If you feel motivated, you could also try creating simulated LLM personas that would have simulated conversation with the agent, similar to [Co-Gym](https://arxiv.org/abs/2412.15701).
- Save both the metric definitions and results.
- To submit this part, I recommend you just link give a link to a branch on your project's github repository.


Below is some example code implementing the same verifier using LLM-as-judge and rule-based python function (though the rule-based method would miss edge cases for sure). The default starter implementation provided below uses the [Judges](https://github.com/quotient-ai/judges) library, but you can also create it from scratch or use another library. Some useful reference links, from the Huggingface Cookbook:
- [Using LLM-as-a-judge 🧑‍⚖️ for an automated and versatile evaluation](https://huggingface.co/learn/cookbook/en/llm_judge) -- A general walkthrough on LLM-as-judge (check it out / see also the documentation)
- [Evaluating AI Search Engines with judges - the open-source library for LLM-as-a-judge evaluators ⚖️](https://huggingface.co/learn/cookbook/en/llm_judge_evaluating_ai_search_engines_with_judges_library) -- More examples on the `Judges` library.

### TODO: Please fill in your intrinsic evaluation design here.

- Metric 1: Front-Loading Discipline

Question Discipline
  - How it's computed: Count the number of question marks in the agent's turn.
  - What it hopes to reflect: Whether the agent is asking multiple questions at the user in a single turn rather than asking one focused question at a time.

Pre-Pause Length
  - How it's computed: Count the number of sentences the agent sends before its first question mark.(it passes if <= 4)
  - What it hopes to reflect: Whether the agent is front loading too much content before inviting the user in.

- Metric 2: Answer first Adherence
  - How it's computed: Using LLM-as-a-judge to check if the user's previous question was a direct question(Requiring a yes/no). Only the yes tturned as scored and on those turns, the judge evaluates whether the agent answered the question before steering toward anything else.
  - What it hopes to reflect: Whether the agent actually addresses what the user asked before redirecting

Then provide a link to a branch on your project's github repository with your implmentation of these metrics.

Note that if you find it challenging to complete this coding part, filling in this section will still allow you to earn partial credit.

In [ ]:
# install the package
!pip install judges
# because judges use Instructor for structured generation under the hood, you'd also want to install
# the right router to the provider.
# see info here: https://python.useinstructor.com/integrations/
!pip install "instructor[bedrock]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.2/71.2 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.5/119.5 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.8 MB/s eta 0:00:00


In [ ]:
from textwrap import dedent
from judges.base import BaseJudge, Judgment
import instructor

class UserInputFormatJudge(BaseJudge):
    """
    A judge that evaluates whether the agent correctly stops to ask users questions.
    """

    def _judge(self, user_prompt: str, system_prompt = None):
      # might need to overwrite this function since the default code
      # misses some important input variable for models
      messages = self._build_messages(user_prompt, system_prompt)
      client = instructor.from_provider(self.model)
      judgment = client.chat.completions.create(
        modelId=self.model.replace("bedrock/", ""),
        messages=messages,
        temperature=0.0,
        response_model=Judgment,
      )
      return judgment.reasoning, judgment.score

    def judge(self, input: str, output: str, expected: str = None) -> Judgment:
      system_prompt = "You are an expert in reading python code."

      user_prompt = dedent(
        f"""
        Evaluate whether the following response correctly called the 'user_input_tool' function and
        the 'final_answer' function consecutively.

        Response to evaluate:
        ```
        {output}
        ```

        Basically, decide whether a 'final_answer' function is always called after a 'user_input_tool' function,
        with the input of final_answer being the output of user_input_tool.
        If no 'user_input_tool' is called, you can simply return 'True'.

        Return 'True' if the response is polite and respectful, 'False' otherwise.
        """
      )
      reasoning, score = self._judge(
        user_prompt=user_prompt,
        system_prompt=system_prompt,
      )
      return Judgment(reasoning=reasoning, score=score, score_type="boolean")


class UserInputFormatJudgeCode(BaseJudge):
  """
  The rule-based version for the same evaluation.
  This is a simplified function for demonstration only
  """

  def judge(self, input: str, output: str, expected: str = None) -> Judgment:
    import ast
    tree = ast.parse(output)
    found_user_input_tool = False
    correct_direct_call = False
    calls = []

    class CallVisitor(ast.NodeVisitor):
      def visit_Call(self, node):
        if isinstance(node.func, ast.Name):
          calls.append(node.func.id)
        elif isinstance(node.func, ast.Attribute):
          calls.append(node.func.attr)  # For obj.method() calls
        self.generic_visit(node)  # Continue visiting children

    CallVisitor().visit(tree)

    if not "user_input_tool" in calls:
      return Judgment(
        reasoning="The user_input_tool was not called",
        score=True, score_type="boolean")
    else:
      idx = calls.index("user_input_tool")
      correct_direct_call = len(calls) > idx + 1 and calls[idx + 1] == "final_answer"
      return Judgment(
          reasoning="Direct call to user_input_tool detected",
          score=correct_direct_call, score_type="boolean")

ModuleNotFoundError: No module named 'judges'

In [ ]:
# Since the judge only needs LLM outputs, we can filter to only have LLM outputs
from smolagents import ActionStep

filtered_model_outputs = []
for step in collab_agent.memory.steps:
  # you can use print(step.to_messages()) to check what's in the responses,
  # or check the documentation here
  if isinstance(step, ActionStep):
    code_calls = [tc.arguments for tc in step.tool_calls if tc.name == "python_interpreter"]
    filtered_model_outputs += code_calls
filtered_model_outputs[0]

'response = user_input_tool(\n    context="Welcome message to start conversation",\n    question="Hello! How can I assist you today?"\n)\nfinal_answer(response)'

In [ ]:
# Initialize your judge
model_id = "us.meta.llama3-3-70b-instruct-v1:0"
user_input_format_judge = UserInputFormatJudge(model=f'bedrock/{model_id}')
user_input_format_judge_code = UserInputFormatJudgeCode(model='')

results = [user_input_format_judge.judge(input="", output=o) for o in filtered_model_outputs]
results_code = [user_input_format_judge_code.judge(input="", output=o) for o in filtered_model_outputs]

In [ ]:
for o, r, rc in zip(filtered_model_outputs, results, results_code):
  print("Model output:")
  print(o)
  print("LLM as judge:")
  print(r)
  print("Code as judge:")
  print(rc)
  print("\n\n\n ------------------------------------ \n\n\n")

Model output:
response = user_input_tool(
    context="Welcome message to start conversation",
    question="Hello! How can I assist you today?"
)
final_answer(response)
LLM as judge:
score=True reasoning="The provided response correctly calls the 'user_input_tool' function and then immediately calls the 'final_answer' function with the output of 'user_input_tool' as its input, following the expected sequence of operations." score_type='boolean'
Code as judge:
score=True reasoning='Direct call to user_input_tool detected' score_type='boolean'



 ------------------------------------ 





In [ ]:
# convert to scores

llm_judge_acc = sum([r.score for r in results]) / len(results)
code_judge_acc = sum([r.score for r in results_code]) / len(results_code)

print(f"LLM as judge accuracy: {llm_judge_acc}")
print(f"Code as judge accuracy: {code_judge_acc}")

LLM as judge accuracy: 1.0
Code as judge accuracy: 1.0


### Final agent vs. 4-16 agent — Experiment Results comparison

This subsection addresses assignment item #4 (run the project evaluation on the **4-16 version** of the agent and submit an Experiment Results section for it). It does **not** replace the intrinsic-evaluation tables you already have above — it adds the head-to-head against the earlier agent.

- **Final agent:** branch `AgentV2.1` @ `f94da78` ([repo](https://github.com/cyrilagbewalikoku-oss/G10COS498_AiUseTutor/tree/AgentV2.1))
- **4-16 agent:** branch `build/agent-2` @ [`7c5492d`](https://github.com/cyrilagbewalikoku-oss/G10COS498_AiUseTutor/commit/7c5492d7d71a7381b2ead3df453627963c296e7e) — last commit on 4-16, the class-feedback day.
- **Full 3–4 page write-up of changes + future work:** [`docs/final-vs-4-16-comparison.md`](../docs/final-vs-4-16-comparison.md).

The cells below load **saved results JSON** from `Experiment Results/` so a grader can read the numbers without re-running the harness or spending tokens.

In [1]:
# Final-vs-4-16 head-to-head from saved JSON (no API spend; no re-execution needed).
import json
from pathlib import Path

RESULTS = Path("../Experiment Results")  # repo-root folder; path is relative to evaluation/

agent416 = json.loads((RESULTS / "agent-4-16-2026-05-14T18-20Z-results.json").read_text())
sim_only = json.loads((RESULTS / "agent-final-2026-05-14T18-20Z-results.json").read_text())

def overall(r, m):
    return r["aggregates"]["overall"][m]

print(f"{'Metric':18s} {'4-16 agent':>22s} {'Final agent':>22s} {'\u0394':>8s}")
print("-" * 74)
for m in ("front_loading", "answer_first"):
    a = overall(agent416, m); b = overall(sim_only, m)
    delta = (b['rate'] or 0) - (a['rate'] or 0)
    print(f"{m:18s} {a['passed']}/{a['applicable']:<3d} ({a['rate']}){'':>5s} "
          f"{b['passed']}/{b['applicable']:<3d} ({b['rate']}){'':>5s} {delta:+.3f}")

print()
print("Per-persona Front-Loading (pass/total):")
print(f"{'persona':25s} {'4-16':>10s} {'Final':>10s}")
print("-" * 50)
for t416, tfin in zip(agent416["transcripts"], sim_only["transcripts"]):
    fl_a = [r for r in t416["turn_results"] if "front_loading" in r]
    fl_b = [r for r in tfin["turn_results"] if "front_loading" in r]
    a_p = sum(1 for r in fl_a if r["front_loading"]["passed"])
    b_p = sum(1 for r in fl_b if r["front_loading"]["passed"])
    print(f"{t416['source_id'].split('/',1)[-1]:25s}    {a_p}/{len(fl_a):<3d}     {b_p}/{len(fl_b):<3d}")

print()
print("Per-persona Answer-First (pass/applicable):")
print(f"{'persona':25s} {'4-16':>10s} {'Final':>10s}")
print("-" * 50)
for t416, tfin in zip(agent416["transcripts"], sim_only["transcripts"]):
    af_a = [r for r in t416["turn_results"] if r.get("answer_first", {}).get("applicable")]
    af_b = [r for r in tfin["turn_results"] if r.get("answer_first", {}).get("applicable")]
    a_p = sum(1 for r in af_a if r["answer_first"]["passed"])
    b_p = sum(1 for r in af_b if r["answer_first"]["passed"])
    print(f"{t416['source_id'].split('/',1)[-1]:25s}    {a_p}/{len(af_a):<3d}     {b_p}/{len(af_b):<3d}")


Metric                         4-16 agent            Final agent        Δ
--------------------------------------------------------------------------
front_loading      9/19  (0.474)      9/19  (0.474)      +0.000
answer_first       8/8   (1.0)      7/10  (0.7)      -0.300

Per-persona Front-Loading (pass/total):
persona                         4-16      Final
--------------------------------------------------
fatigued-returner            4/5       2/5  
novice-curious               5/8       4/8  
skeptical-engineer           0/6       3/6  

Per-persona Answer-First (pass/applicable):
persona                         4-16      Final
--------------------------------------------------
fatigued-returner            0/0       1/1  
novice-curious               6/6       3/5  
skeptical-engineer           2/2       3/4  


**Reading.** Front-Loading is unchanged at the aggregate (9/19 in both), but the per-persona view tells the real story: **`skeptical-engineer` improved from 0/6 to 3/6** after the 4-16 → final prompt edits carved out direct questions from the ANE scaffold (CLAUDE.md changes around the "Answer first" and "Scaffolding Pattern" sections). The flip side is small Front-Loading regressions on `novice-curious` (5/8 → 4/8) and `fatigued-returner` (4/5 → 2/5), driven by **question-stacking** — the final agent sometimes emits 3–4 `?` in one turn when offering choice menus, a side-effect of prompt growth (9.1 KB → 14.3 KB) diluting the "ONE question per message" rule.

Answer-First's apparent 1.00 → 0.70 drop is largely a **denominator-shift artefact**: the persona LLM responds differently to each agent version, so the classifier flagged different applicable-turn counts (8 vs 10). The full causal analysis, future-work plan (mechanical-constraints block, expanded persona corpus, new routing metric), and design rationale are in [`docs/final-vs-4-16-comparison.md`](../docs/final-vs-4-16-comparison.md).

## Step 2 — Extrinsic Evaluation (User Study)

- Recruit >=3 participants who are not in this class.
- Write a user study protocol including:
  - **Evaluation objective**: What aspect are you thinking of evaluating?
  - **Participant type**: e.g., tech-savvy travelers, casual tourists, novice computer users.
  - **Tasks**: What scenario(s) will they complete with your agent? (e.g., plan a weekend trip to a city with budget constraints)
  - **Metrics and analysis**: What will you track and how are you converting them into analysis (e.g., task success rate, time taken, satisfaction rating)?
  - **Baselines** (if any): What are you comparing against?
  - **Interview questions / pre or post survey design**: Short debrief to capture qualitative feedback (e.g., “What part of the interaction was most helpful?”, “What would you change?”)

Run the study and collect both quantitative and qualitative data. **To do so, make sure you use appropriate tools**. e.g. You may write code to collect user clickstreams/logs, use Google Form for survey questions, etc.



Useful links/references for running better studies:
- Competitive Research Methods, Chapter 5, by Elizabeth Goodman, Mike Kuniavsky and Andrea Moed in Observing the User Experience
- [Universal tools- Recruiting and Interviewing, Chapter 6](https://drive.google.com/file/d/1E7CgPEpqxVZmIR0YaaxWbt1NjqpgeICv/view?usp=drive_link) by Elizabeth Goodman, Mike Kuniavsky and Andrea Moed in Observing the User Experience
- [Example Interview Guide- Reading Ahead Interview Guide](https://drive.google.com/file/d/1jFwxgdco0oiaN7wFnSPxvJvL8jgCoeY9/view?usp=drive_link) by Portigal Consulting

### TODO: Please fill in study protocol here.

- **Evaluation objective**: Evaluate how the agent provides information; does it answer your question before asking another? Does the agent only ask one question at a time? Is the agent concise? The agent can ‘demo’ prompts against itself. After asking for prompt guidance, ask it to demo two different prompts.
- **Participant type**: e.g., tech-savvy travelers, casual tourists, novice computer users.
- **Tasks**: Make sure only one question is asked at a time. Make sure that the agent answers the question you provided before asking one of its own questions.
- **Metrics and analysis**: Rate your satisfaction score with how you feel the agent helped you in the goal of improving your AI prompting/usage. If there are more than one question per response, or the agent does NOT answer your question at all, make sure to note that.
- **Baselines** (if any): Optional: compare against any other AI out there, write thoughts and what agent you compared against.
- **Interview questions / pre or post survey design**: What would you change? (anything) Did this agent feel helpful? Was the agent invasive in question?


Please either copy and paste your chat session, or take screenshots of your chat session using print screen or the print feature on the top left dropdown.

**New Evaluation Guide**

- **Evaluation objective**: Perceived improvement on users AI usage
- **Participant type**: e.g., tech-savvy travelers, casual tourists, novice computer users.
- **Tasks**:
1. Give the agent a past experience with an AI agent, and see whether you could've handled prompting differently. Use demo mode if prompted.
2. Ask the AI about good prompting practices, and ask for a demo with bad prompts vs good prompts.
3. Talk to the AI about what are good and bad situations to utilize an AI agent for, and why?
4. Feel free to test your own ideas - the primary purpose of this agent is to help you improve your skill when using AI. Feel free to ask it any questions and judge it's responses within this scope.
- **Metrics and analysis**: Rate your satisfaction score on a scale of 1 to 10, with 1 being poor and 10 being very helpful, on how you feel the agent helped you in the goal of improving your AI prompting/usage. If there are more than one question per response, or the agent does NOT answer your question at all, make sure to note that.
- **Baselines** (if any): Do you use AI now? What for and how often? After using this AI, will you use other AI more or less frequently?
- **Interview questions / pre or post survey design**:

**Post Survey Questions:**
1. Did you feel comfortable with the answers the AI provided?
2. Was there anything about the interface that you felt was out of place or missing?
3. Did you notice any of your own methods that the agent discussed or picked up on?
4. Was the AI aggressive with corrections?
5. On a scale of (1-10), would you use this AI agent without being asked to?
6. Please explain your answer to "Would you use this AI agent, without being asked to?"

****

### **Subject One**

  - **Evaluation objective**: Evaluate how the agent provides information; does it answer your question before asking another? Does the agent only ask one question at a time? Is the agent concise? The agent can ‘demo’ prompts against itself. After asking for prompt guidance, ask it to demo two different prompts.

It does, and continues to sequentially ask one question at a time. The demo-ing feature was clever, and would be good in larger applications of test quizzes, AI literacy evidence, and more.


  - **Participant type**: e.g., tech-savvy travelers, casual tourists, novice computer users.

User familiar with AI LLMs and Agents but curious to learn more

  - **Tasks**: Make sure only one question is asked at a time. Make sure that the agent answers the question you provided before asking one of its own questions.

Done!

  - **Metrics and analysis**: Rate your satisfaction score with how you feel the agent helped you in the goal of improving your AI prompting/usage. If there are more than one question per response, or the agent does NOT answer your question at all, make sure to note that.

I think the agent was successfully able to meet my objectives and questions. There was only one follow up question, and while some answers were delayed until the model had enough information, this proved effective in a more comprehensive result at large.

  - **Baselines** (if any): Optional: compare against any other AI out there, write thoughts and what agent you compared against.

I think this is a more concentrated niche for an AI role, and has very specific use cases for AI education or training. I see this as a positive opportunity to integrate into teaching environments. Given the specificity of the purpose, larger AI usages are less applicable here. I think for this focused area, this AI exceeds expectations, but for more general usage, larger models would be more efficient for those goals.

  - **Interview questions / pre or post survey design**: What would you change? (anything) Did this agent feel helpful? Was the agent invasive in question?

A technical change- add a log out button! It's possible I missed it, but I accidentally logged into someone else's account. I thought the profile selection was AI personas for teaching AI literacy, and didn't realize these were other user's accounts.

**Link to the interaction**
https://drive.google.com/file/d/1fGNMXye-ZjRSEuRPeiXpF9mJs2yUw-fl/view?usp=sharing

****

## **Subject Two:**

- **Evaluation objective**: Evaluate whether the agent encourages responsible and educational AI usage while maintaining a conversational flow. Determine if the agent answers questions directly before asking follow-up questions and whether it keeps interactions focused by asking one question at a time.

The agent maintained a structured and thoughtful conversation throughout the interaction. Each response directly addressed the previous question before introducing a single follow-up prompt, which made the experience feel natural and easy to follow. The pacing worked particularly well for discussing ethical AI use in academic environments, since the agent guided reflection instead of immediately providing definitive answers.

- **Participant type**: e.g., tech-savvy travelers, casual tourists, novice computer users.

Freshman Comuter Science Student

- **Tasks**:Evaluate whether the agent:

- - Answers questions before asking its own
- - Limits follow-up questions to one at a time
- - Encourages critical thinking rather than dependency on AI
- - Provides guidance on ethical AI usage in academics

Successfully completed.

- **Metrics and analysis**: Rate your satisfaction score on a scale of 1 to 10, with 1 being poor and 10 being very helpful, on how you feel the agent helped you in the goal of improving your AI prompting/usage. If there are more than one question per response, or the agent does NOT answer your question at all, make sure to note that.

The agent performed well in balancing guidance with user autonomy. Rather than simply telling the user what to do, it encouraged reflection about learning outcomes and ethical considerations. The responses were concise while still informative, and the conversational structure prevented the interaction from becoming overwhelming. The one-question-at-a-time format made the discussion feel more engaging and less interrogative.

- **Baselines** (if any): Do you use AI now? What for and how often? After using this AI, will you use other AI more or less frequently?

Compared to broader AI chatbots that often prioritize speed and direct answers, this agent focused more on learning-oriented interaction. Its strength came from prompting the user to think critically about how AI should be used rather than treating it purely as a productivity tool. This makes it particularly effective for educational or training scenarios where AI literacy and responsible use are important goals.

- **Interview questions / pre or post survey design**:

**Post Survey Questions:**
1. Did you feel comfortable with the answers the AI provided?
2. Was there anything about the interface that you felt was out of place or missing?
3. Did you notice any of your own methods that the agent discussed or picked up on?
4. Was the AI aggressive with corrections?
5. Would you use this AI outside of this session, without being asked?
6. Why or why not?

The interaction felt helpful and appropriately paced. The agent’s responses were supportive without being overly repetitive or intrusive. One strength was that it reinforced responsible AI practices, such as checking course policies and understanding generated solutions rather than blindly copying them. I would not make significant changes to the interaction style itself, though additional customization for different academic subjects or experience levels could improve flexibility.

**Link to the interaction**
https://drive.google.com/file/d/1SZvNXMkNMPyrGNI0lzHkg_lAmEYTfc2Z/view?usp=sharing

****
## **Subject Three:**

- **Evaluation objective**: Evaluate whether the agent encourages responsible and educational AI usage while maintaining a conversational flow. Determine if the agent answers questions directly before asking follow-up questions and whether it keeps interactions focused by asking one question at a time.

The agent performed well as an instructional tool for improving prompting skills. Instead of instantly correcting the user’s request, it encouraged the user to think about why the original prompt lacked detail and scope management. The conversation remained focused and organized, with the agent asking only one follow-up question at a time while still directly addressing the user’s responses.

- **Participant type**: e.g., tech-savvy travelers, casual tourists, novice computer users.

Freshman Comuter Science Student

- **Tasks**:Evaluate whether the agent:

- - Answers questions before asking its own
- - Limits follow-up questions to one at a time
- - Encourages critical thinking rather than dependency on AI
- - Provides guidance on ethical AI usage in academics

It did well.

- **Metrics and analysis**: Rate your satisfaction score on a scale of 1 to 10, with 1 being poor and 10 being very helpful, on how you feel the agent helped you in the goal of improving your AI prompting/usage. If there are more than one question per response, or the agent does NOT answer your question at all, make sure to note that.

The interaction was effective because the agent focused on teaching reasoning rather than simply providing the “correct” answer. By comparing the AI request to instructions given to a human freelancer, the agent made the importance of context and specificity easy to understand. The gradual narrowing of the prompt into a single task also reinforced useful prompting habits without making the experience feel overly technical.

- **Baselines** (if any): Do you use AI now? What for and how often? After using this AI, will you use other AI more or less frequently?

Many general-purpose AI systems immediately rewrite weak prompts without explaining the underlying issue. In contrast, this agent emphasized learning and self-improvement, which makes it more useful in educational environments or AI literacy training. While broader AI tools may be faster for direct productivity tasks, this agent excels at helping users develop stronger long-term prompting skills.

- **Interview questions / pre or post survey design**:

**Post Survey Questions:**
1. Did you feel comfortable with the answers the AI provided?
2. Was there anything about the interface that you felt was out of place or missing?
3. Did you notice any of your own methods that the agent discussed or picked up on?
4. Was the AI aggressive with corrections?
5. Would you use this AI outside of this session, without being asked?
6. Why or why not?

The interaction felt helpful and appropriately paced throughout. The questioning style encouraged reflection without becoming repetitive or intrusive. One potential improvement could be allowing the agent to eventually show a fully optimized example prompt after the guided learning process, giving users both the reasoning process and a polished final example to compare against.

**Link to the interaction**
https://drive.google.com/file/d/1nZZHTPnZa1yySAE_B7fZiwrygwBZvc9c/view?usp=sharing

****

- **Evaluation objective**: Evaluate how the agent provides information; does it answer your question before asking another? Does the agent only ask one question at a time? Is the agent concise? The agent can ‘demo’ prompts against itself. After asking for prompt guidance, ask it to demo two different prompts.
- **Participant type**: e.g., tech-savvy travelers, casual tourists, novice computer users.
- **Tasks**: Make sure only one question is asked at a time. Make sure that the agent answers the question you provided before asking one of its own questions.
- **Metrics and analysis**: Rate your satisfaction score with how you feel the agent helped you in the goal of improving your AI prompting/usage. If there are more than one question per response, or the agent does NOT answer your question at all, make sure to note that.
- **Baselines** (if any): Optional: compare against any other AI out there, write thoughts and what agent you compared against.
- **Interview questions / pre or post survey design**: What would you change? (anything) Did this agent feel helpful? Was the agent invasive in question?


Please either copy and paste your chat session, or take screenshots of your chat session using print screen or the print feature on the top left dropdown.

****


In [ ]:
## Write code to analyze your data here.

# load data...

# see raw patterns...

# get visualization...

# Run stats...

## Conclusion...

# Just for fun, an example of a data analysis agent and some "agent recipies"
# i.e. patterns other people have created:
# https://huggingface.co/learn/cookbook/en/agent_data_analyst

## Step 3: Summarize & Reflect

Here, you will complete your writeup by reflecting on your evaluation, in these three parts:


### **Evaluation overview**

Create a SPHERE Evaluation Card for your evaluation. To do so,
- Carefully read [this paper](https://arxiv.org/abs/2504.07971) (you can ignore the appendix except for the examples starting on p.24)
- Go to [this website](https://sphere-eval.github.io/) and create the card by filling in the corresponding textboxes.
- Put the exported Json here:
The full machine-readable JSON is committed under [`docs/sphere-card-intrinsic.json`](../docs/sphere-card-intrinsic.json). Inline summary:

  ```json
  {
    "sysname": "SAGE — Scaffolded AI Guidance for Engagement",
    "card_type": "intrinsic",
    "aspects": {
      "S": "Pedagogical tutor for AI-agent literacy; 18 skills under .claude/skills/; runtime is Streamlit + CLI on Anthropic SDK. In scope: question discipline, front-loading, answer-first on direct questions. Out of scope: tool-call correctness, factual accuracy of content.",
      "P": "Intrinsic: 3 persona-simulated learners (novice-curious, skeptical-engineer, fatigued-returner) + 7 hand-authored gold transcripts (4 positive, 3 negative). Extrinsic: 3 university students outside the team.",
      "H": {
        "metric_1": "Front-Loading Discipline (rule-based, evaluation/metrics/front_loading.py): per turn, ?-count <= 1 AND <= 4 sentences before first ?.",
        "metric_2": "Answer-First Adherence (LLM-as-judge, evaluation/metrics/answer_first.py): Haiku 4.5 classifier (yes_no | factual | open | not_a_question) then Sonnet 4.6 grader (answered_first | answered_and_followed_up | redirected_without_answer | non_answer)."
      },
      "E": {
        "final_agent_full_corpus":    { "front_loading": "23/53 (0.434)", "answer_first": "9/13 (0.692)" },
        "final_agent_simulated_only": { "front_loading": "9/19 (0.474)",  "answer_first": "7/10 (0.700)" },
        "4_16_agent_simulated_only":  { "front_loading": "9/19 (0.474)",  "answer_first": "8/8 (1.000)"  },
        "discriminative_validity":    "Authored positive (0.500) > negative (0.333) Front-Loading rate."
      },
      "R": [
        "Small corpus (19 simulated SAGE turns per agent version) makes per-persona rates fragile.",
        "Conversation-flow divergence: each agent version drives the persona LLM into different applicable-turn sets, confounding direct comparison on Metric 2.",
        "Single LLM grader (no human-judge alignment study yet)."
      ],
      "Empirical_strength": "medium"
    },
    "saved_artifacts": [
      "Experiment Results/final-agent-2026-05-14T18-15Z-results.json",
      "Experiment Results/agent-final-2026-05-14T18-20Z-results.json",
      "Experiment Results/agent-4-16-2026-05-14T18-20Z-results.json"
    ]
  }
  ```

### **Experiment Results**

Summarize your learning from each evaluation, and propose one more evaluation you could think of running for this agent.

1. Intrinsic evaluation : Question discipline
  - Describe what was evaluated:Counting the number of "?" in each turn
  - Present the key results (tables, figures, or example outputs):

  
| Label      | Front-Loading |
|------------|----------------|
| negative   | 6/18 (0.333)   |
| positive   | 8/16 (0.5)     |
| simulated  | 9/19 (0.474)   |


  - Write at least one conclusion you can draw from the study:
  
  a. The agent violated either the number of questions to ask or keeping the senctence length <=4 over 50% of the time. The failures could come from either having multiple questions in a turn or having more than four sentences of explanation before the question.
  

2. Intrinsic evaluation Answer First Adherence
  - Describe what was evaluated:Checks if the agent answers the user query first before moving onto to teaching/coaching.
  - Present the key results (tables, figures, or example outputs):
| Label      |  Answer-First |
|------------|----------------|
| negative   |  2/2 (1.0)     |
| positive   |  0/1 (0.0)     |
| simulated  |  7/10 (0.7)    |

  - Write at least one conclusion you can draw from the study:
  
  a. When a learner asks a direct yes/no or factual question, SAGE answers before coaching about 69% of the time.



3. Exrinsic evaluation General Flow and functionality
  - Describe what was evaluated: User tested various features, from the demo-ing to purposeful testing of quality.
  - Present the key results (tables, figures, or example outputs): https://drive.google.com/file/d/1fGNMXye-ZjRSEuRPeiXpF9mJs2yUw-fl/view?usp=sharing

  Use link for output of conversation

  - Write at least one conclusion you can draw from the study: While the functionality of the agent is there, it seems to be a very niche role in the AI space - we believe that niche purposes for AI is good and the direction AI will go. As there are more and more specific agents for specific things, standing out in your respective field is important.

## **Potential evaluation that could also be run**
- Briefly describe this evaluation: LLm as a judge checks if anything the learner pastes sensitive data (shares something that sgouldn't be shared with an agent)
- Explain why it’s relevant: Since our agent teaches about responsible AI use, sharing seneitive data is one of the wrong things to do with an LLM and our agent teaches against that.
- Specify if it’s intrinsic or extrinsic: **Intrinsic**
- Outline the metric or study design:

- We will use a rule based filter as the first checker for the input.
- Classify the type of input and add a "grade" to show the level of sensitivity.
- Based on the sensitivity/risk, it turns to coach the user on the risks.


### **Additional insights**

(Optional, but I would love to hear your insights and reflection on this process evaluating the agent.)

# Submission

1. Make sure all the important execution results are saved as files and then correctly displayed in whatever file you turn in, in a way that would allow me to grade you without re-running code.
2. Submit a link to your readme and/or evaluation write-up file in your repo (or a google doc)
3. Submit on brightspace a link to your write-up (in markdown in your project github repo, or in this notebook).

**Forgetting about this will result in deduction in the clarity and effort scores.**

# Submissions

The seven submission items from the assignment brief. Existing chat transcripts (Subjects 1–3 in the extrinsic section above) and intrinsic results in Step 3 are unchanged; this cell only adds the links and references the assignment asks for.

1. **Project specification** — `docs/architecture.md`, [`docs/technical-report.md`](../docs/technical-report.md). Agent contract: [`CLAUDE.md`](../CLAUDE.md).

2. **Prior iteration report (+ final-sprint addendum)** — [`docs/roadmap.md`](../docs/roadmap.md) for the prior plan; [`docs/final-vs-4-16-comparison.md` §3–§4](../docs/final-vs-4-16-comparison.md) for what *was* and *was not* accomplished after the 4-23 feedback.

3. **Completed final project evaluation (latest agent)** — this notebook's Step 3 cells above, plus the saved results files committed under [`Experiment Results/`](../Experiment%20Results/):
   - [`final-agent-2026-05-14T18-15Z-results.json`](../Experiment%20Results/final-agent-2026-05-14T18-15Z-results.json) — full run (7 authored + 3 simulated, 53 SAGE turns)
   - [`final-agent-2026-05-14T18-15Z-summary.md`](../Experiment%20Results/final-agent-2026-05-14T18-15Z-summary.md) — human-readable summary

4. **4-16 agent Experiment Results** — see "Final agent vs. 4-16 agent" section earlier in this notebook. Commit: [`7c5492d`](https://github.com/cyrilagbewalikoku-oss/G10COS498_AiUseTutor/commit/7c5492d7d71a7381b2ead3df453627963c296e7e) on branch `build/agent-2`. Saved results:
   - [`agent-4-16-2026-05-14T18-20Z-results.json`](../Experiment%20Results/agent-4-16-2026-05-14T18-20Z-results.json)
   - [`agent-4-16-2026-05-14T18-20Z-summary.md`](../Experiment%20Results/agent-4-16-2026-05-14T18-20Z-summary.md)
   - 4-16 system prompt fixture: [`evaluation/fixtures/prompts/sage-prompt-4-16-7c5492d.txt`](fixtures/prompts/sage-prompt-4-16-7c5492d.txt)
   - 4-16 simulated transcripts: [`evaluation/fixtures/simulated/4-16-agent/`](fixtures/simulated/4-16-agent/)

5. **3–4 page write-up (latest vs 4-16 comparison + future work + plan)** — [`docs/final-vs-4-16-comparison.md`](../docs/final-vs-4-16-comparison.md). §1–§3 cover per-metric, per-persona deltas with hypotheses for what changed and why; §4 covers ~1.5 pages of future work across three concrete proposals (mechanical-constraints block, expanded persona corpus, new routing metric) with design rationale and plans.

6. **GitHub repo** — <https://github.com/cyrilagbewalikoku-oss/G10COS498_AiUseTutor/> (final branch: [`AgentV2.1`](https://github.com/cyrilagbewalikoku-oss/G10COS498_AiUseTutor/tree/AgentV2.1)).

7. **3–4 page technical summary** — [`docs/technical-report.md`](../docs/technical-report.md).

---

**Reproducibility.** All numbers shown in this notebook are loaded from JSON committed under `Experiment Results/`, so the grader can read every table without re-running the harness. The full pipeline if a re-run is desired:

```bash
# Final agent — uses sage.prompts.SYSTEM_PROMPT
python -m evaluation.personas.simulator --all
python -m evaluation.run_evaluation --label final-agent

# 4-16 agent — same code, different system prompt
python -m evaluation.personas.simulator --all \
  --sage-prompt evaluation/fixtures/prompts/sage-prompt-4-16-7c5492d.txt
mv evaluation/fixtures/simulated/*.json evaluation/fixtures/simulated/4-16-agent/
python -m evaluation.run_evaluation \
  --simulated-dir evaluation/fixtures/simulated/4-16-agent \
  --no-authored --label agent-4-16
```

53 unit tests under `evaluation/tests/` exercise parser, metrics, and judge cache without any API spend (`pytest evaluation/tests -v`).
